MACS 30113 \
Yuzhao (Jerry) Jin; CNETID: yuzhaojin1 \
Duo (Doris) Wang; CNETID: dorisduowang

Full Visualizations on cleaned and analyzed data only

In [ ]:
import io, os
import boto3
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from dotenv import load_dotenv

_here = Path().resolve()
for _candidate in [_here / ".env",
                   _here.parent / ".env"]:
    if _candidate.exists():
        load_dotenv(_candidate)
        break

BUCKET = os.environ["BUCKET"]

TEAL    = "#1D9E75"
TEAL_LT = "#9FE1CB"
BLUE    = "#378ADD"
CORAL   = "#D85A30"
PURPLE  = "#534AB7"
PUR_LT  = "#7F77DD"
PUR_DK  = "#3C3489"
GRAY    = "#888780"
SRC_CLR = {"gdelt": TEAL, "guardian": BLUE, "nyt": CORAL, "fomc": PURPLE}

plt.rcParams.update({
    "font.family":      "DejaVu Sans",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.grid":        True,
    "grid.alpha":       0.22,
    "grid.linestyle":   "--",
    "figure.dpi":       130,
})

s3 = boto3.client("s3")

def read_s3_csv(key):
    return pd.read_csv(io.BytesIO(
        s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()))

def read_s3_parquet(key):
    return pd.read_parquet(io.BytesIO(
        s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()))

def upload(path, s3_key):
    s3.upload_file(str(path), BUCKET, s3_key)
    print(f"Uploaded → s3://{BUCKET}/{s3_key}")

OUT = Path("figures"); OUT.mkdir(exist_ok=True)

In [ ]:
# Load all processed outputs
print("Loading from S3 ...")
summary     = read_s3_csv("processed/lead_lag/summary.csv")
benchmarks  = read_s3_csv("processed/benchmarks/benchmarks.csv")
sentiment   = read_s3_parquet("processed/sentiment/sentiment.parquet")
per_meeting = read_s3_parquet("processed/lead_lag/per_meeting.parquet")
cosine      = read_s3_parquet("processed/similarity/cosine.parquet")

sentiment["doc_date"]      = pd.to_datetime(sentiment["doc_date"])
per_meeting["meeting_date"] = pd.to_datetime(per_meeting["meeting_date"])
cosine["meeting_date"]      = pd.to_datetime(cosine["meeting_date"])

print(f"summary:     {summary.shape}")
print(f"benchmarks:  {benchmarks.shape}")
print(f"sentiment:   {sentiment.shape}  sources={sentiment['source'].value_counts().to_dict()}")
print(f"per_meeting: {per_meeting.shape}")
print(f"cosine:      {cosine.shape}")


In [ ]:
# Figure 1: GDELT lead-lag Pearson r (main finding)
gdelt = summary[summary["source"] == "gdelt"].sort_values("lag_days")
lags  = gdelt["lag_days"].tolist()
r     = gdelt["pearson_r_vol"].tolist()
pvals = gdelt["p_value"].tolist()
sig   = [p < 0.05 for p in pvals]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(lags, r,
       color=[TEAL if s else TEAL_LT for s in sig],
       width=0.65, zorder=3)

peak_idx = r.index(max(r))
ax.annotate(f"r = {max(r):.3f}\n(p < 0.001)",
            xy=(lags[peak_idx], r[peak_idx]),
            xytext=(lags[peak_idx]+0.45, r[peak_idx]+0.045),
            fontsize=9, color=TEAL,
            arrowprops=dict(arrowstyle="->", color=TEAL, lw=1.2))

for lag, rv, p in zip(lags, r, pvals):
    stars = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else ""
    if stars:
        ax.text(lag, rv+0.01, stars, ha="center", va="bottom", fontsize=9, color=TEAL)

ax.axhline(0, color=GRAY, lw=0.8)
ax.set_xlabel("Lag days relative to FOMC meeting", fontsize=11)
ax.set_ylabel("Pearson r  (cosine similarity vs S&P volatility)", fontsize=11)
ax.set_title("GDELT: Lead-lag correlation with S&P 500 volatility\n"
             "(n = 33 per lag, 2015–2025)", fontsize=12, pad=10)
ax.set_xticks(lags)
ax.set_xticklabels([f"+{l}" if l>0 else ("meeting\nday" if l==0 else str(l)) for l in lags])
ax.set_ylim(-0.05, 0.72)
ax.legend(handles=[mpatches.Patch(color=TEAL, label="Significant (p<0.05)"),
                   mpatches.Patch(color=TEAL_LT, label="Not significant")],
          fontsize=9)
fig.tight_layout()
fig.savefig(OUT/"fig1_lead_lag_gdelt.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Figure 2: Cosine similarity at lag 0 by source
lag0    = summary[summary["lag_days"]==0].sort_values("mean_cosine", ascending=True)
sources = lag0["source"].tolist()
cosines = lag0["mean_cosine"].tolist()
colors  = [SRC_CLR.get(s, GRAY) for s in sources]

fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.barh(sources, cosines, color=colors, height=0.5, zorder=3)
ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=10)
ax.set_xlabel("Mean cosine similarity to FOMC minutes", fontsize=11)
ax.set_title("Textual similarity at meeting day (lag = 0)", fontsize=12, pad=8)
ax.set_xlim(0, 0.30); ax.grid(axis="y", alpha=0)
fig.tight_layout()
fig.savefig(OUT/"fig2_cosine_by_source.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Figure 3 Benchmark: Python vs NumPy vs Numba
impls    = benchmarks["impl"].tolist()
seconds  = benchmarks["seconds"].tolist()
baseline = seconds[0]
speedups = [baseline/s for s in seconds]
xlabels  = ["Pure\nPython", "NumPy\nvectorised", "Numba JIT\n@jit(parallel)"]
colors   = [PUR_DK, PUR_LT, PURPLE]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.8))

bars1 = ax1.bar(xlabels, seconds, color=colors, width=0.5, edgecolor="black", lw=0.7)
for bar, t, sp in zip(bars1, seconds, speedups):
    lbl = f"{t:.3f}s" if sp==1 else f"{t:.3f}s\n({sp:.1f}×)"
    ax1.text(bar.get_x()+bar.get_width()/2, t+0.005, lbl,
             ha="center", va="bottom", fontsize=9, fontweight="bold")
ax1.set_ylabel("Time (s) — lower is better", fontsize=10)
ax1.set_title("Absolute time\n5M token workload", fontsize=10)
ax1.set_ylim(0, max(seconds)*1.40); ax1.grid(axis="y", alpha=0.3)

bars2 = ax2.bar(xlabels, speedups, color=colors, width=0.5, edgecolor="black", lw=0.7)
ax2.axhline(1, color=GRAY, lw=0.8, ls="--", alpha=0.6)
for bar, sp in zip(bars2, speedups):
    ax2.text(bar.get_x()+bar.get_width()/2, sp+0.04, f"{sp:.1f}×",
             ha="center", va="bottom", fontsize=10, fontweight="bold")
ax2.set_ylabel("Speedup vs Python (×) — higher is better", fontsize=10)
ax2.set_title("Speedup ratio", fontsize=10)
ax2.set_ylim(0, max(speedups)*1.28); ax2.grid(axis="y", alpha=0.3)

fig.suptitle("Numba JIT Benchmark: Python vs NumPy vs Numba", fontsize=12, y=1.01)
fig.tight_layout()
fig.savefig(OUT/"fig3_benchmark.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Figure 4: All sources across all lags
all_lags = sorted(summary["lag_days"].unique())
src_cfg  = [("gdelt",    TEAL,  "-",  "o", "GDELT  (n=33)"),
            ("guardian", BLUE,  "--", "s", "Guardian  (n=5–62)"),
            ("nyt",      CORAL, ":",  "^", "NYT  (n=6–22)")]

fig, ax = plt.subplots(figsize=(10, 4.5))
for src, color, ls, marker, lbl in src_cfg:
    df_s = summary[summary["source"]==src].set_index("lag_days")
    xs = [l for l in all_lags if l in df_s.index]
    ys = [df_s.loc[l, "pearson_r_vol"] for l in xs]
    ax.plot(xs, ys, color=color, ls=ls, marker=marker,
            markersize=5, lw=1.8, label=lbl, zorder=3)

ax.axvspan(-3.4, 1.4, alpha=0.06, color=TEAL, label="GDELT significant window")
ax.axhline(0, color=GRAY, lw=0.8)
ax.axvline(0, color=GRAY, lw=0.6, ls="--", alpha=0.5)
ax.set_xlabel("Lag days relative to FOMC meeting", fontsize=11)
ax.set_ylabel("Pearson r  (vs S&P 500 volatility)", fontsize=11)
ax.set_title("Lead-lag correlation: all media sources  (2015–2025)", fontsize=12, pad=8)
ax.set_xticks(all_lags)
ax.set_xticklabels([f"+{l}" if l>0 else str(l) for l in all_lags], fontsize=9)
ax.legend(fontsize=9, framealpha=0.9, loc="upper left")
fig.tight_layout()
fig.savefig(OUT/"fig4_all_sources_lags.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Figure 5: FOMC Hawkishness Timeline 2015–2025
fomc = (sentiment[sentiment["source"]=="fomc"]
        .sort_values("doc_date")
        .reset_index(drop=True))

dates = fomc["doc_date"]
vals  = fomc["net_per_1k"]

fig, ax = plt.subplots(figsize=(13, 4.5))

# Shade hawkish (positive) vs dovish (negative) regions
ax.fill_between(dates, vals, 0,
                where=vals >= 0, alpha=0.18, color=CORAL,
                label="Hawkish (net_per_1k > 0)")
ax.fill_between(dates, vals, 0,
                where=vals < 0,  alpha=0.18, color=BLUE,
                label="Dovish (net_per_1k < 0)")
ax.plot(dates, vals, color=PURPLE, lw=1.5, zorder=3)
ax.scatter(dates, vals,
           c=[CORAL if v >= 0 else BLUE for v in vals],
           s=35, zorder=5, edgecolors="white", linewidths=0.4)

ax.axhline(0, color=GRAY, lw=0.9, ls="--", alpha=0.6)

# Annotate key events
events = {
    "2020-03-15": ("COVID\nEmergency cut", -0.5, "bottom"),
    "2022-03-16": ("Rate hike\ncycle begins", 1.5, "top"),
    "2022-11-02": ("Peak\nhawkishness", 1.5, "top"),
    "2019-09-18": ("Fed pivots\ndovish", -0.5, "bottom"),
}
for date_str, (txt, yoffset, va) in events.items():
    dt = pd.to_datetime(date_str)
    yv = fomc.loc[fomc["doc_date"]==dt, "net_per_1k"].values
    if len(yv):
        ax.annotate(txt, xy=(dt, yv[0]),
                    xytext=(dt, yv[0] + yoffset*8),
                    ha="center", va=va, fontsize=7.5,
                    arrowprops=dict(arrowstyle="-", color=GRAY, lw=0.7))

ax.set_xlabel("FOMC meeting date", fontsize=11)
ax.set_ylabel("Net hawkishness per 1,000 tokens", fontsize=11)
ax.set_title("FOMC Minutes Hawkishness Over Time  (88 meetings, 2015–2025)\n"
             "Positive = more hawkish words; Negative = more dovish words",
             fontsize=12, pad=8)
ax.legend(fontsize=9, loc="upper left")
fig.tight_layout()
fig.savefig(OUT/"fig5_fomc_hawkishness_timeline.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Figure 6: S&P 500 Volatility Pre vs Post Meeting
# One row per meeting: vol_pre, vol_post, mean cosine across all lags
vol_df = (per_meeting
          .groupby("meeting_date")
          .agg(vol_pre=("vol_pre","first"),
               vol_post=("vol_post","first"),
               mean_cosine=("cosine_sim","mean"))
          .reset_index())
vol_df["vol_change"] = vol_df["vol_post"] - vol_df["vol_pre"]
vol_df["year"] = vol_df["meeting_date"].dt.year

fig, ax = plt.subplots(figsize=(7, 6))

sc = ax.scatter(vol_df["vol_pre"], vol_df["vol_post"],
                c=vol_df["mean_cosine"], cmap="YlOrRd",
                s=60, edgecolors="black", linewidths=0.5,
                zorder=4, alpha=0.85)

# y = x reference line
lim = max(vol_df[["vol_pre","vol_post"]].max()) * 1.05
ax.plot([0, lim], [0, lim], color=GRAY, lw=1.0, ls="--",
        label="y = x  (no change)")
ax.fill_between([0, lim], [0, lim], [lim, lim],
                alpha=0.04, color=CORAL, label="Volatility ↑ post-meeting")
ax.fill_between([0, lim], [0, 0], [0, lim],
                alpha=0.04, color=BLUE,  label="Volatility ↓ post-meeting")

# Label notable outliers
for _, row in vol_df.nlargest(3, "vol_post").iterrows():
    ax.annotate(row["meeting_date"].strftime("%Y-%m"),
                xy=(row["vol_pre"], row["vol_post"]),
                xytext=(5, 3), textcoords="offset points",
                fontsize=7.5, color="black")

cb = fig.colorbar(sc, ax=ax, shrink=0.7)
cb.set_label("Mean cosine similarity (media coverage)", fontsize=9)

ax.set_xlabel("Realised volatility 7 days PRE-meeting", fontsize=11)
ax.set_ylabel("Realised volatility 7 days POST-meeting", fontsize=11)
ax.set_title("S&P 500 Volatility: Pre vs Post FOMC Meeting\n"
             "Colour = media coverage intensity (cosine similarity)",
             fontsize=12, pad=8)
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.legend(fontsize=8.5, loc="upper left")
fig.tight_layout()
fig.savefig(OUT/"fig6_volatility_pre_post.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Figure 7: GDELT Cosine Heatmap
gd_pivot = (cosine[cosine["source"]=="gdelt"]
            .pivot(index="meeting_date", columns="lag_days", values="cosine_sim")
            .sort_index())

# Sort chronologically and format row labels
gd_pivot.index = pd.to_datetime(gd_pivot.index)
gd_pivot = gd_pivot.sort_index()
row_labels = [d.strftime("%Y-%m") for d in gd_pivot.index]

fig, ax = plt.subplots(figsize=(9, 10))
im = ax.imshow(gd_pivot.values, aspect="auto",
               cmap="YlOrRd", vmin=0, vmax=gd_pivot.values.max())

ax.set_xticks(range(len(gd_pivot.columns)))
ax.set_xticklabels(
    [f"+{c}" if c>0 else ("0\n(meeting)" if c==0 else str(c))
     for c in gd_pivot.columns],
    fontsize=10)
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=8)

ax.set_xlabel("Lag day relative to FOMC meeting", fontsize=11)
ax.set_ylabel("FOMC meeting date", fontsize=11)
ax.set_title("GDELT Cosine Similarity to FOMC Minutes\n"
             "Heatmap across all 33 covered meetings × 7 lag days",
             fontsize=12, pad=10)

cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.01)
cb.set_label("Cosine similarity", fontsize=9)

# Draw vertical line at lag -2 (peak)
ax.axvline(gd_pivot.columns.tolist().index(-2) - 0.5,
           color=TEAL, lw=1.5, ls="--", alpha=0.7)
ax.axvline(gd_pivot.columns.tolist().index(-2) + 0.5,
           color=TEAL, lw=1.5, ls="--", alpha=0.7)
ax.text(gd_pivot.columns.tolist().index(-2), -1.2, "lag −2\n(peak r)",
        ha="center", va="top", fontsize=8, color=TEAL)

fig.tight_layout()
fig.savefig(OUT/"fig7_gdelt_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Figure 8: Sentiment Distribution by Source
# Non-zero only for meaningful distribution (GDELT is 83% zero-inflated)
nonzero = sentiment[sentiment["net_per_1k"] != 0].copy()
src_order = ["fomc", "nyt", "guardian", "gdelt"]
src_labels_map = {"fomc":"FOMC\nMinutes",
                  "nyt":"New York\nTimes",
                  "guardian":"The\nGuardian",
                  "gdelt":"GDELT\n(non-zero)"}

fig, ax = plt.subplots(figsize=(9, 4.5))
data_by_src = [nonzero[nonzero["source"]==s]["net_per_1k"].clip(-100, 100)
               for s in src_order]
colors_bp = [SRC_CLR[s] for s in src_order]

bp = ax.boxplot(data_by_src,
                patch_artist=True,
                notch=True,
                medianprops=dict(color="black", lw=2),
                whiskerprops=dict(lw=1.2),
                capprops=dict(lw=1.2),
                flierprops=dict(marker=".", markersize=2, alpha=0.3))

for patch, color in zip(bp["boxes"], colors_bp):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Annotate medians
for i, (src, data) in enumerate(zip(src_order, data_by_src)):
    med = data.median()
    ax.text(i+1, med+2.5, f"med={med:.1f}", ha="center", fontsize=8)

ax.axhline(0, color=GRAY, lw=0.9, ls="--", alpha=0.6)
ax.set_xticklabels([src_labels_map[s] for s in src_order], fontsize=10)
ax.set_ylabel("Net hawkishness per 1,000 tokens\n(clipped to ±100)", fontsize=11)
ax.set_title("Sentiment Distribution by Source  (non-zero documents only)\n"
             "Positive = hawkish; Negative = dovish",
             fontsize=12, pad=8)
ax.grid(axis="y", alpha=0.25)

# Add n= labels below each box
n_counts = [len(d) for d in data_by_src]
for i, n in enumerate(n_counts):
    ax.text(i+1, ax.get_ylim()[0]+2, f"n={n:,}", ha="center",
            fontsize=7.5, color=GRAY)

fig.tight_layout()
fig.savefig(OUT/"fig8_sentiment_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Figure 9: Top 10 Most Media-Aligned FOMC Meetings
# Mean cosine across all lags and sources per meeting
top10 = (per_meeting
         .groupby("meeting_date")["cosine_sim"]
         .mean()
         .sort_values(ascending=False)
         .head(10)
         .reset_index())
top10["label"] = top10["meeting_date"].dt.strftime("%Y-%m-%d")
top10["year"]  = top10["meeting_date"].dt.year

# Color by year using a gradient
year_min, year_max = top10["year"].min(), top10["year"].max()
norm = mcolors.Normalize(vmin=year_min, vmax=year_max)
cmap = plt.cm.get_cmap("plasma")
bar_colors = [cmap(norm(y)) for y in top10["year"]]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(range(len(top10)), top10["cosine_sim"],
               color=bar_colors, height=0.65, edgecolor="black", lw=0.5)
ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=9)

ax.set_yticks(range(len(top10)))
ax.set_yticklabels(top10["label"], fontsize=10)
ax.invert_yaxis()   # highest at top
ax.set_xlabel("Mean cosine similarity to FOMC minutes (all lags)", fontsize=11)
ax.set_title("Top 10 Most Media-Aligned FOMC Meetings\n"
             "Colour = year (darker = more recent)",
             fontsize=12, pad=8)
ax.set_xlim(0, top10["cosine_sim"].max() * 1.22)
ax.grid(axis="x", alpha=0.25)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cb = fig.colorbar(sm, ax=ax, shrink=0.6, aspect=20)
cb.set_label("Year", fontsize=9)
cb.set_ticks([year_min, year_max])
cb.set_ticklabels([str(year_min), str(year_max)])

fig.tight_layout()
fig.savefig(OUT/"fig9_top10_meetings.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Figure 10 Direct Scatter: vol_change vs cosine at lag −2
# GDELT at lag -2
lag2 = per_meeting[(per_meeting["source"]=="gdelt") &
                   (per_meeting["lag_days"]==-2)].copy()
lag2["vol_change"] = lag2["vol_post"] - lag2["vol_pre"]
lag2["year"] = lag2["meeting_date"].dt.year

x = lag2["cosine_sim"].values.astype(float)
y = lag2["vol_change"].values.astype(float)

# Pearson r and regression line
r_val, p_val = stats.pearsonr(x, y)
slope, intercept, *_ = stats.linregress(x, y)
x_line = np.linspace(x.min(), x.max(), 100)
y_line = slope * x_line + intercept

fig, ax = plt.subplots(figsize=(7, 5.5))

# Color by year
norm = mcolors.Normalize(vmin=lag2["year"].min(), vmax=lag2["year"].max())
sc = ax.scatter(x, y, c=lag2["year"], cmap="coolwarm",
                s=55, edgecolors="black", lw=0.5, zorder=4, alpha=0.85)

ax.plot(x_line, y_line, color=TEAL, lw=2, ls="-",
        label=f"OLS fit  (r = {r_val:.3f}, p < 0.001)", zorder=5)
ax.axhline(0, color=GRAY, lw=0.8, ls="--", alpha=0.5)

ax.set_xlabel("GDELT cosine similarity at lag −2 days", fontsize=11)
ax.set_ylabel("Volatility change  (vol_post − vol_pre)", fontsize=11)
ax.set_title(
    "Direct scatter: media alignment 2 days before FOMC meeting\n"
    "vs post-meeting S&P 500 volatility change",
    fontsize=12, pad=8)

# Pearson r annotation box
ax.text(0.96, 0.95,
        f"Pearson r = {r_val:.3f}\np = {p_val:.4f}\nn = {len(x)}",
        transform=ax.transAxes,
        ha="right", va="top", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                  edgecolor=TEAL, alpha=0.9))

ax.legend(fontsize=9)
cb = fig.colorbar(sc, ax=ax, shrink=0.7)
cb.set_label("FOMC meeting year", fontsize=9)

fig.tight_layout()
fig.savefig(OUT/"fig10_scatter_lag2.png", dpi=150, bbox_inches="tight")
plt.show()


## Upload all figures to S3

In [ ]:
figs = [
    "fig1_lead_lag_gdelt.png",
    "fig2_cosine_by_source.png",
    "fig3_benchmark.png",
    "fig4_all_sources_lags.png",
    "fig5_fomc_hawkishness_timeline.png",
    "fig6_volatility_pre_post.png",
    "fig7_gdelt_heatmap.png",
    "fig8_sentiment_distribution.png",
    "fig9_top10_meetings.png",
    "fig10_scatter_lag2.png",
]
for fname in figs:
    p = OUT / fname
    if p.exists():
        upload(p, f"processed/figures/{fname}")
    else:
        print(f"Missing: {fname} — run its cell first")
